In [32]:
#!/usr/bin/env python
"""
pyCaverDock Workflow

This script implements a workflow where you supply a receptor PDB file and a ligand file.
It creates a working subdirectory (named after the receptor), moves the PDB there,
modifies the CAVER configuration on the fly (adjusting first_frame, last_frame, and time_sparsity),
runs CAVER on the copied PDB (using the workdir as the -pdb option), and then for each tunnel found,
starts a separate CaverDock run using pyCaverDock.

Dependencies:
  - pyCaverDock (which provides classes like CaverDock, Receptor, Ligand, load_tunnel, discretize_tunnel, etc.)
  - pandas, matplotlib, py3Dmol, MDAnalysis
  - openbabel (via pybel)
"""

import os
import shutil
import subprocess
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob

# Import pyCaverDock API
from pycaverdock import (
    CaverDock,
    Receptor,
    Ligand,
    load_tunnel,
    discretize_tunnel,
    Direction,
    TrajectoryType,
    EnergyProfilePlot,
    plot_results
)
from pycaverdock.experiment import convert_eprofile_analysis
from pycaverdock.utils import get_basename

def prepare_protein(input_file):
    """Convert a PDB receptor to PDBQT using OpenBabel."""
    if os.path.splitext(input_file)[1].lower() != ".pdbqt":
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
        conv = ob.OBConversion()
        conv.SetInFormat("pdb")
        conv.SetOutFormat("pdbqt")
        conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
        obmol = ob.OBMol()
        conv.ReadFile(obmol, input_file)
        obmol.CorrectForPH()  # optional
        obmol.AddHydrogens()
        conv.WriteFile(obmol, output_file)
        return output_file
    return input_file

def prepare_ligand(input_file):
    """Convert a ligand (SDF or MOL2) to PDBQT using OpenBabel."""
    ext = os.path.splitext(input_file)[1].lower()[1:]
    base = os.path.splitext(input_file)[0]
    output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()
    mol.make3D()
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

class Miner:
    def __init__(self, caver_path=None):
        """
        Initialize with paths to the CAVER executables.
        If not provided, they are assumed to be in subdirectories "caver" and "caverdock".
        """
        if caver_path:
            self.caver_path = caver_path  
        else:
            try:
                self.caver_path = script_path = os.path.abspath(__file__)
            except:
                self.caver_path = "caver/"

    
    def run_caver(self, receptor_file, first_frame=1, last_frame=1, time_sparsity=1):
        """
        Run CAVER on the receptor PDB using a temporary directory in the original working directory.

        This method:
        • Creates a temporary working directory in the script's original working directory.
        • Copies the receptor PDB into that directory.
        • Reads and modifies the default CAVER config file to set first_frame, last_frame, and time_sparsity.
        • Runs CAVER with -pdb set to the working directory.
        • Cleans up the temporary directory after execution.
        """
        # Get the directory where the script was originally run
        original_cwd = os.getcwd()
        
        # Create a temporary directory within the original working directory
        temp_dir = tempfile.mkdtemp(dir=original_cwd)
        print(temp_dir)
        print(self.caver_path)
        
        try:
            # Copy the receptor file into temp_dir
            receptor_path = os.path.join(temp_dir, os.path.basename(receptor_file))
            shutil.copy(receptor_file, temp_dir)

            # Read the default config file (assumed to be in the same dir as caver.jar)
            default_config_path = f"{self.caver_path}/config_default.txt"
            with open(default_config_path, "r") as f:
                config_lines = f.readlines()

            # Modify parameters: first_frame, last_frame, time_sparsity
            new_config_lines = []
            for line in config_lines:
                stripped = line.strip()
                if stripped.startswith("time_sparsity"):
                    new_config_lines.append(f"time_sparsity {time_sparsity}\n")
                elif stripped.startswith("first_frame"):
                    new_config_lines.append(f"first_frame {first_frame}\n")
                elif stripped.startswith("last_frame"):
                    new_config_lines.append(f"last_frame {last_frame}\n")
                else:
                    new_config_lines.append(line)

            # Save modified config file in temp_dir
            new_config_path = os.path.join(temp_dir, "config.txt")
            with open(new_config_path, "w") as f:
                f.writelines(new_config_lines)

            # Build the CAVER command
            cmd = [
                "java", "-jar", f"{self.caver_path}/caver.jar",
                "-Xmx1200m",
                "-home", self.caver_path,
                "-pdb", f"{temp_dir}/",
                "-conf", new_config_path,
                "-out", f"{temp_dir}/"
            ]
            print("Running CAVER:", " ".join(cmd))
            subprocess.check_call(cmd)

            # Assume tunnels are saved as "tunnels.pdb" in temp_dir
            tunnels_file = os.path.join(temp_dir, "tunnels.pdb")
            return tunnels_file, temp_dir

        finally:
            print("hi")
            # Clean up the temporary directory after execution
            #shutil.rmtree(temp_dir, ignore_errors=True)

    def run_caverdock(self, receptor_pdbqt, ligand_pdbqt, tunnel, work_dir, output_subdir, **kwargs):
        """
        Run a CaverDock analysis (via pyCaverDock) on a given tunnel.
        The tunnel parameter is expected to be a discretized tunnel (e.g., from discretize_tunnel).
        The output_subdir is created under work_dir for this tunnel’s results.
        """
        os.makedirs(output_subdir, exist_ok=True)
        # Instantiate the pyCaverDock API object
        cd_api = CaverDock()
        # Create a descriptive title using the receptor base name and tunnel index
        title = f"{get_basename(receptor_pdbqt)}_{os.path.basename(output_subdir)}"
        print(f"Running CaverDock for {title}")
        trajectory = cd_api.run(
            receptor=receptor_pdbqt,
            ligand=ligand_pdbqt,
            tunnel=tunnel,
            direction=Direction.OUT,
            trajectory_type=TrajectoryType.LOWERBOUND,
            mpi_processes=4,
            seed=42,
            exhaustiveness=8,
            workdir=output_subdir,
            title=title
        )
        return trajectory

    def parse_energy_profile(self, profile_file):
        df = pd.read_csv(profile_file, delim_whitespace=True, names=["coordinate", "energy"])
        return df

    def plot_energy_profile(self, energy_df, output_png):
        plt.figure(figsize=(8, 6))
        plt.plot(energy_df["coordinate"], energy_df["energy"], marker='o')
        plt.xlabel("Tunnel coordinate")
        plt.ylabel("Energy (kcal/mol)")
        plt.title("CaverDock Energy Profile")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(output_png)
        plt.close()
        print(f"Energy profile plot saved to {output_png}")

    def visualize_tunnel(self, receptor_file, tunnel_file, output_html):
        view = py3Dmol.view(width=800, height=600)
        with open(receptor_file, "r") as f:
            pdb_data = f.read()
        view.addModel(pdb_data, "pdb")
        with open(tunnel_file, "r") as f:
            tunnel_data = f.read()
        view.addModel(tunnel_data, "pdb")
        view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
        view.setStyle({'model': 1}, {'stick': {'color': 'red', 'radius': 0.3}})
        view.zoomTo()
        html_str = view._make_html()
        with open(output_html, "w") as f:
            f.write(html_str)
        print(f"Interactive visualization HTML saved to {output_html}")

if __name__ == "__main__":
    receptor_pdb = "1be0.pdb"    # Supply your receptor PDB file
    ligand_file = "EtOH.sdf"      # Supply your ligand file (SDF or MOL2)
    # Adjust first_frame, last_frame, and time_sparsity as desired.
    miner = Miner(caver_path="caver")
    miner.run_caver("1be0.pdb")


/home/erikna/compchem/WhatCat/miner/tmp6azkt48q
caver
Running CAVER: java -jar caver/caver.jar -Xmx1200m -home caver -pdb /home/erikna/compchem/WhatCat/miner/tmp6azkt48q/ -conf /home/erikna/compchem/WhatCat/miner/tmp6azkt48q/config.txt -out /home/erikna/compchem/WhatCat/miner/tmp6azkt48q/
hi


Exception in thread "main" java.lang.NullPointerException: Cannot invoke "java.io.File.exists()" because "d" is null
	at caver.util.FileOperations.safeDir(FileOperations.java:33)
	at caver.util.FileOperations.safeSubfile(FileOperations.java:68)
	at caver.util.FileOperations.safeSub(FileOperations.java:52)
	at caver.ui.Launcher.main(Launcher.java:1383)


CalledProcessError: Command '['java', '-jar', 'caver/caver.jar', '-Xmx1200m', '-home', 'caver', '-pdb', '/home/erikna/compchem/WhatCat/miner/tmp6azkt48q/', '-conf', '/home/erikna/compchem/WhatCat/miner/tmp6azkt48q/config.txt', '-out', '/home/erikna/compchem/WhatCat/miner/tmp6azkt48q/']' returned non-zero exit status 1.

In [ ]:
#TODO error is missing start coordinates. these need to be written to the caver config file
#TODO add clustering methods from miner.py

NameError: name '__file__' is not defined